# E4d: adaptive-crop CNN

**Revision plan, 00 log section 19 (approved 17 Oct 2026).** Implements what both reviewers suggested: Reviewer 2 point 5
("justify or **implement adaptive cropping**") and Reviewer 4 point 8 ("**use adaptive crops** or stratify by size"; the
size strata are in E47).

**Adaptive crop:** around the same tumour centre as before, a cube with side **max(96, 1.2 × the tumour's largest extent)**
voxels, resized to 96³ (linear for CT, nearest for masks); liver masking and normalisation unchanged. Tumours that fit in
96³ get exactly the same crop as before (checked). The CNN is trained with the **phase-dropout protocol of E3** (same folds,
seeds, inner validation split). It is reported **next to** the fixed-crop phase-dropout CNN, which stays the main CNN.

**Version 3 (19 Oct 2026):** v2 also stopped without an error message in the HCC-TACE-Seg step (after 50 of 103 patients; memory 13 of 179 GB, so not memory). v3 changes only the order: MCT-LTDiag, the internal crops, training and the internal + MCT-LTDiag evaluation run first and save their results; HCC-TACE-Seg runs last, one patient at a time, with a per-patient log (`E4d_tace_progress.log`) and a crash log in the output folder. Method, cells and rules are unchanged.

**Version 2 (18 Oct 2026):** the first run stopped without an error message in the HCC-TACE-Seg step (Cell 9), most likely out of memory: large tumours need adaptive cubes of 300–600 voxels before resizing. v2 builds these cubes in float32 from the needed region only (identical crops), processes 3 HCC-TACE-Seg patients at a time instead of 8, and shows memory use in the progress lines. Nothing else changed.

**Order (v3):** E6's cells prepare MCT-LTDiag (203; with the E8.0 grid fix), the internal adaptive crops are built from the
original scans, the CNN is trained and evaluated internally and on MCT-LTDiag (results saved), then E5c's cells prepare
HCC-TACE-Seg (corrected) and it is evaluated last.
Gates: the original pipelines must reproduce their saved crops/predictions.

**Runtime:** **A100 GPU**, about 2.5–3 hours (reading 68 GB of MCT-LTDiag scans again, internal scans, 5 trainings).
**Resume:** set `RESUME_RUN_ID` in Cell 1 to the stopped run's time and run all again. **When it finishes,** Cell 15 saves everything to Drive and shuts the runtime down, so no compute units are wasted (set `SHUTDOWN_WHEN_DONE = False` in Cell 15 to keep it). **Safety:** outputs go to a new folder
`liver_project/revision_2026-10/E4d_<time>/`.

In [ ]:
# Cell 1 — Setup (E6's cell): installs, paths, input checks, output folder E4d_<time>
import os, sys, io, re, json, time, shutil, tarfile, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e6_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'pydicom', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E6_TEST_ROOT'])
    WORK = Path(os.environ['E6_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
import pydicom
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from scipy.stats import beta, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
N_IO = 6 if IN_COLAB else 2                       # patients read/preprocessed at the same time
N_JOBS = os.cpu_count() if IN_COLAB else 1        # parallel radiomics processes

INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'mct_patient_table': (MCT_DIR / 'metadata' / 'meta_info_patient.csv', 'f9f0fc730b1ec75cecb615304aa0f69db30225fb559b460ef78d0fe693220f0d'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
LARGE_INPUTS = {'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320)}
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def md5_of(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists() or (IN_COLAB and path.stat().st_size != size):
        problems.append(f'missing or changed size: {path}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'pydicom': getattr(pydicom, '__version__', 'unknown'),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE), 'cpus': os.cpu_count()}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESUME_RUN_ID = None   # to continue a stopped run, put its time here, e.g. '20261018T020304Z'
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
OUT_DIR = REVISION_DIR / f'E4d_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))
print('Inputs and models checked:', len(INPUTS) + len(LARGE_INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Frozen models must reproduce their saved internal test predictions; internal orientation reference
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E6_TEST_PROCESSED'])

split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
TEST_CT, TEST_LABEL = {}, {}
for pid in test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        TEST_CT[pid], TEST_LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])

def zscore_nonzero_per_phase(x4):          # Week 4, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

@torch.no_grad()
def cnn_probs(models, channels, arrays):
    """Mean softmax over the 5 fold models; arrays: patient -> uint8 (4, 96, 96, 96) in internal axis order."""
    rows = {}
    for pid, ct in arrays.items():
        x = torch.from_numpy(zscore_nonzero_per_phase(ct)[channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

def macro_auc3(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

y_test = [TEST_LABEL[p] for p in test_ids]
gate = {}
for name, key, model_tag in [('original_CNN', 'e3_original_predictions', 'original'),
                             ('phase_dropout_CNN', 'e3_d_predictions', 'D_phase_dropout')]:
    saved = pd.read_csv(INPUTS[key][0], dtype={'patient_id': str})
    saved = saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']]
    ours = cnn_probs(*CNNS[name], {p: TEST_CT[p] for p in test_ids}).loc[test_ids]
    gate[name] = {'max_prob_diff': float(np.abs(ours.to_numpy() - saved.to_numpy()).max()),
                  'macro_auc_ours': macro_auc3(y_test, ours.to_numpy()), 'macro_auc_saved': macro_auc3(y_test, saved.to_numpy())}

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
pool_cols = rad_models[0]['feature_pool']
X_int_test = internal.set_index('patient_id').loc[test_ids, pool_cols].to_numpy(dtype=np.float32)
p_int = np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X_int_test[:, m['sel_idx']])))
                 for m in rad_models], axis=0)
saved_int = pd.read_csv(INPUTS['e2_noshape_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
gate['radiomics_noshape'] = {'max_prob_diff': float(np.abs(p_int - saved_int).max())}
GATE_OK = (all(gate[n]['max_prob_diff'] < 0.02 and abs(gate[n]['macro_auc_ours'] - gate[n]['macro_auc_saved']) < 0.005
               for n in ['original_CNN', 'phase_dropout_CNN']) and gate['radiomics_noshape']['max_prob_diff'] < 1e-6)
print('Internal reproduction check:', json.dumps(gate, indent=1))
if IN_COLAB and not GATE_OK:
    raise RuntimeError('STOP — the frozen models do not reproduce their saved internal test predictions.')

# ---- internal orientation reference (stored arrays, as Week 2 read them with nibabel) ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254                               # >= ~198 HU on the -200..200 -> 0..255 scale

def orientation_descriptor(ct0255_ijk, liver_ijk):
    """Where the liver and the bone are in an (i, j, k) array, as fractions of each axis (no class labels)."""
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')   # non-contrast: bone not confused with contrast

def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}

with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
internal_desc.insert(0, 'patient_id', p_rows['patient_id'].tolist())
internal_desc.to_csv(OUT_DIR / 'E4d_internal_orientation_reference.csv', index=False)
ok = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok['bone_j'] - .5) > abs(ok['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok['bulk_k'] < .5).mean()), 'n': int(len(ok)),
       'median_liver_i': float(ok['liver_i'].median()), 'median_bone_j': float(ok['bone_j'].median()),
       'median_bulk_k': float(ok['bulk_k'].median())}
print('Internal orientation reference:', json.dumps(REF, indent=1))
# Must agree with the 12-patient check in the 00 log (17.9): i = left-right with the liver at low i, spine at high j, bulk at low k
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')

In [ ]:
# Cell 3 — E6's facts check for all 203 MCT-LTDiag scans, with the E8.0 grid fix
SEL = pd.read_csv(INPUTS['mct_selected'][0], dtype=str)
SEL['size'] = SEL['size'].astype(int)
counts = SEL['type'].value_counts().to_dict()
if IN_COLAB and counts != {'HCC': 103, 'ICC': 100}:
    raise RuntimeError(f'STOP — unexpected MCT-LTDiag selection {counts}.')
WANTED = {'NIFTI/nc.nii.gz': 'P', 'NIFTI/art.nii.gz': 'C1', 'NIFTI/pvp.nii.gz': 'C2', 'NIFTI/delay.nii.gz': 'C3',
          'mask_pvp.nii.gz': 'tumor', 'liver_mask_pvp.nii.gz': 'liver'}
DICOM_FIELDS = ['SeriesDescription', 'Manufacturer', 'ManufacturerModelName', 'SliceThickness', 'PixelSpacing', 'KVP',
                'ConvolutionKernel', 'SeriesInstanceUID']
TAR_TMP, NII_DIR = WORK / 'tar', WORK / 'nifti'
TAR_TMP.mkdir(exist_ok=True)
NII_DIR.mkdir(exist_ok=True)

def window_0255(hu):                          # PLC-CECT release convention: -200..200 HU -> 0..255
    return ((np.clip(hu, -200.0, 200.0) + 200.0) / 400.0 * 255.0).astype(np.float32)

def ijk(img):                                 # SimpleITK array (k, j, i) -> stored (i, j, k) order, as nibabel / Week 2
    arr = sitk.GetArrayFromImage(img)
    return np.ascontiguousarray(np.transpose(arr, (2, 1, 0)))

def grid(img):
    return img.GetSize(), img.GetSpacing(), img.GetOrigin(), img.GetDirection()

def same_grid(a, b):
    return (tuple(a[0]) == tuple(b[0]) and np.allclose(a[1], b[1], rtol=1e-3) and np.allclose(a[2], b[2], atol=0.5)
            and np.allclose(a[3], b[3], atol=1e-3))

def onto(img, reference, is_mask):
    return sitk.Resample(img, reference, sitk.Transform(), sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear,
                         0 if is_mask else -1024)

def nifti_facts(folder):
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    ref = grid(imgs['C2'])
    f = {'size': list(ref[0]), 'spacing': [round(float(s), 4) for s in ref[1]], 'direction': [round(float(d), 3) for d in ref[3]],
         'off_grid': [k for k in imgs if k != 'C2' and not same_grid(grid(imgs[k]), ref)]}
    mm = sitk.MinimumMaximumImageFilter()
    for k in PHASE_ORDER:
        mm.Execute(imgs[k])
        sample = sitk.GetArrayViewFromImage(imgs[k])[::2, ::4, ::4]
        f[f'{k}_min'], f[f'{k}_max'] = float(mm.GetMinimum()), float(mm.GetMaximum())
        f[f'{k}_p50'] = float(np.percentile(sample, 50))
    mins, maxs = [f[f'{k}_min'] for k in PHASE_ORDER], [f[f'{k}_max'] for k in PHASE_ORDER]
    f['intensity_kind'] = ('HU' if max(mins) <= -500 else '0-255' if min(mins) >= 0 and max(maxs) <= 255 else 'unexpected')
    tumor_img = imgs['tumor'] if 'tumor' not in f['off_grid'] else onto(imgs['tumor'], imgs['C2'], is_mask=True)   # E8.0 fix
    liver_img0 = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    tumor, liver = sitk.GetArrayFromImage(tumor_img), sitk.GetArrayFromImage(liver_img0)
    f['tumor_labels'] = [float(v) for v in np.unique(tumor)[:10]]
    f['liver_labels'] = [float(v) for v in np.unique(liver)[:10]]
    f['tumor_voxels'], f['liver_voxels'] = int((tumor > 0).sum()), int((liver > 0).sum())
    f['tumor_components'] = int(ndimage.label(tumor > 0)[1])
    f['tumor_volume_ml'] = f['tumor_voxels'] * float(np.prod(ref[1])) / 1000.0
    f['tumor_outside_liver_fraction'] = float(((tumor > 0) & (liver == 0)).sum() / max(1, (tumor > 0).sum()))
    plain = imgs['P'] if 'P' not in f['off_grid'] else onto(imgs['P'], imgs['C2'], is_mask=False)
    liver_img = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    ct = ijk(plain)
    descriptor = orientation_descriptor(window_0255(ct) if f['intensity_kind'] == 'HU' else ct, ijk(liver_img) > 0)
    if descriptor is None or not np.isfinite(descriptor['bone_i']):
        raise RuntimeError('liver mask or bone too small to determine orientation')
    f.update(descriptor)
    f['nib_equals_sitk_ijk'] = bool(np.allclose(load_nii(folder / 'P.nii.gz'), ijk(imgs['P']).astype(np.float32), atol=1e-3))
    return f

def stage(row):
    pid = row['patient_id']
    folder = NII_DIR / pid
    done = folder / 'facts.json'
    if done.exists():
        return json.loads(done.read_text())
    folder.mkdir(exist_ok=True)
    facts, series = {'patient_id': pid, 'technical_failure': ''}, {}
    local = TAR_TMP / row['name']
    try:
        shutil.copyfile(MCT_DIR / f"tar_{row['type']}" / row['name'], local)
        if md5_of(local) != row['md5']:
            raise RuntimeError('checksum mismatch after copying from Drive')
        found = set()
        with tarfile.open(local, 'r:') as tf:
            for m in tf:
                if not m.isfile():
                    continue
                name = m.name[2:] if m.name.startswith('./') else m.name
                if name in WANTED:
                    (folder / f'{WANTED[name]}.nii.gz').write_bytes(tf.extractfile(m).read())   # fixed local names only
                    found.add(name)
                elif name.startswith('DICOM/'):
                    key = name.rsplit('/', 1)[0]
                    if key not in series:
                        try:
                            ds = pydicom.dcmread(io.BytesIO(tf.extractfile(m).read()), stop_before_pixels=True, force=True)
                            series[key] = {field: str(getattr(ds, field, '')) for field in DICOM_FIELDS}
                        except Exception as error:
                            series[key] = {'error': repr(error)[:100]}
                        series[key]['files'] = 0
                    series[key]['files'] += 1
        if set(WANTED) - found:
            raise RuntimeError(f'missing in tar: {sorted(set(WANTED) - found)}')
        facts.update(nifti_facts(folder))
        if facts['tumor_voxels'] == 0:
            facts['technical_failure'] = 'empty tumour mask'
    except Exception as error:
        facts['technical_failure'] = repr(error)[:200]
    finally:
        local.unlink(missing_ok=True)
    facts['dicom_series'] = list(series.values())
    done.write_text(json.dumps(facts, default=float))
    return facts

t0 = time.time()
rows = [r for _, r in SEL.iterrows()]
FACTS = []
with ThreadPoolExecutor(N_IO) as pool:
    for n, f in enumerate(pool.map(stage, rows), start=1):
        FACTS.append(f)
        if n % 20 == 0 or n == len(rows):
            print(f'  facts check {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

dicom_rows = [{'patient_id': f['patient_id'], **s} for f in FACTS for s in f.get('dicom_series', [])]
pd.DataFrame(dicom_rows).to_csv(OUT_DIR / 'E4d_mct_dicom_series.csv', index=False)
facts_df = pd.DataFrame([{k: v for k, v in f.items() if k != 'dicom_series'} for f in FACTS])
TECH_FAIL = facts_df.loc[facts_df['technical_failure'] != '', ['patient_id', 'technical_failure']]
INCLUDED = facts_df.loc[facts_df['technical_failure'] == '', 'patient_id'].tolist()
kinds = Counter(facts_df.loc[facts_df['technical_failure'] == '', 'intensity_kind'])
print(f'Included {len(INCLUDED)}/{len(SEL)}; technical failures: {TECH_FAIL.to_dict("records")}')
print('Intensity kind:', dict(kinds), '| phases off the portal-venous grid:',
      int(facts_df['off_grid'].map(lambda v: len(v) if isinstance(v, list) else 0).gt(0).sum()), 'patients')
if kinds.get('unexpected', 0):
    raise RuntimeError('STOP — some scans are neither HU nor 0-255 (rule 17.3-1). Send this output.')
if IN_COLAB and not facts_df.loc[facts_df['technical_failure'] == '', 'nib_equals_sitk_ijk'].all():
    raise RuntimeError('STOP — SimpleITK (i, j, k) arrays differ from nibabel arrays.')

# ---- orientation decisions (amendment 17.9-1): match each scan to the internal anatomy ----
def decide(d):
    # front-back axis = the in-plane axis on which the bone (spine) is off-centre; internally that is j
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

ORIENT = {f['patient_id']: decide(f) for f in FACTS if f['patient_id'] in INCLUDED}
orient_df = pd.DataFrame.from_dict(ORIENT, orient='index')
orientation_summary = {}
for part in ['transpose', 'flip_i', 'flip_j', 'flip_k']:
    confident = orient_df[orient_df[f'conf_{part}']]
    majority = bool(confident[part].mean() > .5) if len(confident) else False
    share = float((confident[part] == majority).mean()) if len(confident) else 0.0
    unclear = orient_df.index[~orient_df[f'conf_{part}']]
    orient_df.loc[unclear, part] = majority
    orientation_summary[part] = {'applied_to': int(orient_df[part].sum()), 'unclear_set_to_majority': int(len(unclear)),
                                 'majority': majority, 'agreement_among_clear': round(share, 3)}
    if len(unclear) > 0.10 * len(orient_df):                 # scans may differ from each other; too many unclear ones may not
        raise RuntimeError(f'STOP — orientation part {part!r} is unclear for {len(unclear)} scans (>10%). Send this output.')
ORIENT = orient_df[['transpose', 'flip_i', 'flip_j', 'flip_k']].astype(bool).to_dict('index')
orient_df.to_csv(OUT_DIR / 'E4d_mct_orientation_decisions.csv')
facts_df.to_csv(OUT_DIR / 'E4d_mct_facts.csv', index=False)

ok = facts_df[facts_df['technical_failure'] == '']
dicom = pd.DataFrame(dicom_rows)
FACTS_SUMMARY = {
    'in_plane_spacing_mm': [float(ok['spacing'].map(lambda s: s[0]).min()), float(ok['spacing'].map(lambda s: s[0]).median()),
                            float(ok['spacing'].map(lambda s: s[0]).max())],
    'slice_spacing_mm': [float(ok['spacing'].map(lambda s: s[2]).min()), float(ok['spacing'].map(lambda s: s[2]).median()),
                         float(ok['spacing'].map(lambda s: s[2]).max())],
    'tumour_components': ok['tumor_components'].value_counts().sort_index().to_dict(),
    'tumour_volume_ml_median': float(ok['tumor_volume_ml'].median()),
    'manufacturers': dicom['Manufacturer'].value_counts().to_dict() if 'Manufacturer' in dicom else {},
    'dicom_slice_thickness': dicom['SliceThickness'].value_counts().head(6).to_dict() if 'SliceThickness' in dicom else {},
    'kvp': dicom['KVP'].value_counts().head(6).to_dict() if 'KVP' in dicom else {},
}
print('Facts:', json.dumps(FACTS_SUMMARY, indent=1, default=str))
print('Orientation decisions (amendment 17.9-1):', json.dumps(orientation_summary, indent=1))

In [ ]:
# Cell 4 — E6 preprocessing, primary geometry only, plus the adaptive crop (plan 19, E4d)
INPLANE_MM = 400.0 / 512                       # internal physical pixel size (512 matrix, 400 mm field of view)
CROP_SIZE = 96
VARIANTS = ['primary', 'adaptive']
CROP_DIR = WORK / 'crops'
CROP_DIR.mkdir(exist_ok=True)

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def resample(img, spacing, is_mask):
    size = [max(1, int(round(sz * sp / ts))) for sz, sp, ts in zip(img.GetSize(), img.GetSpacing(), spacing)]
    f = sitk.ResampleImageFilter()
    f.SetOutputSpacing(tuple(float(s) for s in spacing))
    f.SetSize(size)
    f.SetOutputOrigin(img.GetOrigin())
    f.SetOutputDirection(img.GetDirection())
    f.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    f.SetDefaultPixelValue(0 if is_mask else -1024)
    return f.Execute(img)

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

def crops_from(arrays):
    liver, tumor = arrays['liver'] > 0, arrays['tumor'] > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    ct4 = np.stack([np.clip(crop_cube(arrays[p] * liver, center, CROP_SIZE), 0, 255).astype(np.uint8) for p in PHASE_ORDER])
    tumor_crop = (crop_cube(tumor.astype(np.uint8), center, CROP_SIZE) > 0).astype(np.uint8)
    meta = {'shape': list(tumor.shape), 'center': list(center), 'tumor_in_crop': float(tumor_crop.sum() / max(1, tumor.sum()))}
    return ct4, tumor_crop, meta

ADAPT_MARGIN = 1.2
ADAPT_OUT = 96          # own constant: the E5c cells later redefine the crop size as a tuple
def ram_text():                     # memory in use, shown in the progress lines (E4d v2)
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f' | RAM {vm.used / 1e9:.0f}/{vm.total / 1e9:.0f} GB'
    except Exception:
        return ''

def adaptive_side(masks):
    extent = 0
    for m in masks:
        if m.any():
            idx = np.argwhere(m)
            extent = max(extent, int((idx.max(axis=0) - idx.min(axis=0) + 1).max()))
    return max(ADAPT_OUT, int(np.ceil(ADAPT_MARGIN * extent)))

def adaptive_crop(volume, center, side, is_mask):
    if side == ADAPT_OUT:
        return crop_cube(volume, center, side)                     # Week 2 crop, unchanged
    half = side // 2                                               # same cube as crop_cube(volume, center, side),
    starts = [c - half for c in center]                            # built in float32 from the needed region only
    ends = [c + side - half for c in center]                       # (memory: large tumours need cubes of 300-600 voxels)
    region = np.asarray(volume[tuple(slice(max(0, a), min(volume.shape[i], b)) for i, (a, b) in enumerate(zip(starts, ends)))],
                        dtype=np.float32)
    pad = [(max(0, -a), max(0, b - volume.shape[i])) for i, (a, b) in enumerate(zip(starts, ends))]
    cropped = np.pad(region, pad, mode='constant') if any(q != (0, 0) for q in pad) else region
    out = ndimage.zoom(cropped, ADAPT_OUT / side, order=0 if is_mask else 1)
    del region, cropped
    fixed = np.zeros((ADAPT_OUT,) * 3, np.float32)
    sl = tuple(slice(0, min(ADAPT_OUT, s)) for s in out.shape)
    fixed[sl] = out[sl]
    return fixed

def adaptive_crops_from(arrays):
    liver, tumor = arrays['liver'] > 0, arrays['tumor'] > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    side = adaptive_side([tumor])
    ct4 = np.stack([np.clip(adaptive_crop(arrays[p] * liver, center, side, False), 0, 255).astype(np.uint8) for p in PHASE_ORDER])
    tumor_crop = (adaptive_crop(tumor.astype(np.uint8), center, side, True) > 0).astype(np.uint8)
    return ct4, tumor_crop, {'side': side, 'center': list(center)}

def preprocess(pid):
    target = CROP_DIR / f'{pid}.npz'
    meta_file = CROP_DIR / f'{pid}.json'
    if target.exists() and meta_file.exists():
        return json.loads(meta_file.read_text())
    folder = NII_DIR / pid
    facts = next(f for f in FACTS if f['patient_id'] == pid)
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    for k in facts['off_grid']:                                  # rule 17.3-2
        imgs[k] = onto(imgs[k], imgs['C2'], is_mask=k in ('tumor', 'liver'))
    hu = facts['intensity_kind'] == 'HU'
    sz = imgs['C2'].GetSpacing()[2]
    settings = {'primary': (INPLANE_MM, INPLANE_MM, sz)}
    out, meta = {}, {}
    for variant, spacing in settings.items():
        vols = imgs if spacing is None else {k: resample(v, spacing, k in ('tumor', 'liver')) for k, v in imgs.items()}
        if variant == 'S5_week10g_axes':
            arrays = {k: sitk.GetArrayFromImage(v).astype(np.float32) for k, v in vols.items()}   # (k, j, i), as Week 10g
        else:
            arrays = {k: apply_orientation(ijk(v).astype(np.float32), ORIENT[pid]) for k, v in vols.items()}
        for p in PHASE_ORDER:
            arrays[p] = window_0255(arrays[p]) if hu else arrays[p]
        out[f'{variant}_ct'], out[f'{variant}_tumor'], meta[variant] = crops_from(arrays)
        out['adaptive_ct'], out['adaptive_tumor'], meta['adaptive'] = adaptive_crops_from(arrays)
    np.savez_compressed(target, **out)
    meta_file.write_text(json.dumps(meta))
    return meta

t0 = time.time()
CROP_META = {}
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, meta) in enumerate(zip(INCLUDED, pool.map(preprocess, INCLUDED)), start=1):
        CROP_META[pid] = meta
        if n % 20 == 0 or n == len(INCLUDED):
            print(f'  preprocessed {n}/{len(INCLUDED)} ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)

crop_rows = [{'patient_id': pid, 'variant': v, **m} for pid, metas in CROP_META.items() for v, m in metas.items()]
pd.DataFrame(crop_rows).to_csv(OUT_DIR / 'E4d_mct_crop_geometry.csv', index=False)
(OUT_DIR / 'crops_adaptive_mct').mkdir(exist_ok=True)
for pid in INCLUDED:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        np.savez_compressed(OUT_DIR / 'crops_adaptive_mct' / f'{pid}.npz', ct=z['adaptive_ct'], tumor_mask=z['adaptive_tumor'])
geo = pd.DataFrame(crop_rows)
print('Tumour fraction inside the fixed 96^3 crop (median):', geo.loc[geo['variant'] == 'primary', 'tumor_in_crop'].median(),
      '| adaptive side > 96 for', int((geo.loc[geo['variant'] == 'adaptive', 'side'] > ADAPT_OUT).sum()), 'patients')

In [ ]:
# Cell 5 — Keep the MCT-LTDiag crops in memory; gate: primary crops must equal the E6 run's saved crops
E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
E8A_RUN = REVISION_DIR / 'E8a_20261017T162800Z'
if len(INCLUDED) != 203 and IN_COLAB:
    raise RuntimeError(f'STOP — {len(INCLUDED)} MCT-LTDiag patients included, expected 203.')
MCT_IDS = list(INCLUDED)
MCT_ADAPT, MCT_PRIM, MCT_SIDE = {}, {}, {}
for pid in MCT_IDS:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        MCT_ADAPT[pid], MCT_PRIM[pid] = z['adaptive_ct'], z['primary_ct']
    MCT_SIDE[pid] = CROP_META[pid]['adaptive']['side']
diffs = []
for pid in MCT_IDS:
    saved = E6_RUN / 'crops_primary' / f'{pid}.npz'
    saved = saved if saved.exists() else E8A_RUN / 'crops_primary_restored2' / f'{pid}.npz'
    with np.load(saved) as z:
        diffs.append(int(np.abs(z['ct'].astype(int) - MCT_PRIM[pid].astype(int)).max()))
GATE_MCT_CROPS = {'patients_identical': int(sum(d == 0 for d in diffs)), 'max_abs_diff': int(max(diffs))}
print('MCT-LTDiag primary crops vs saved E6/E8.0 crops:', GATE_MCT_CROPS)
if IN_COLAB and GATE_MCT_CROPS['max_abs_diff'] > 1:
    raise RuntimeError('STOP — the MCT-LTDiag preprocessing does not reproduce the saved E6 crops. Send this output.')

In [ ]:
# Cell 11 — Internal adaptive crops from the original scans (Week 2 rules + adaptive side), gate against processed crops
import random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
NUM_EPOCHS = int(os.environ.get('E4D_TEST_EPOCHS', 40))
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 4 if IN_COLAB else 0
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
else:
    PROCESSED_DIR = Path(os.environ['E6_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
LABEL, FIXED, SIZE = {}, {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        LABEL[pid], FIXED[pid], SIZE[pid] = int(z['label']), z['ct'].astype(np.uint8), float(z['tumor_vol_portal'])
table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
table = table[table['patient_id'].isin(dev_ids + test_ids)].drop_duplicates(['patient_id', 'phase'])

def internal_adaptive(pid):
    rows = table[table['patient_id'] == pid].set_index('phase')
    vols = {}
    for ph in PHASE_ORDER:
        ct = load_nii(DATA_DIR / rows.loc[ph, 'ct_path'])
        tumor = load_nii(DATA_DIR / rows.loc[ph, 'mask_path'])
        liver = load_nii(DATA_DIR / rows.loc[ph, 'liver_mask_path'])
        vols[ph] = (ct * (liver > 0).astype(np.float32), tumor > 0, liver > 0)
    side = adaptive_side([v[1] for v in vols.values()])
    out = []
    for ph in PHASE_ORDER:
        ct_masked, tumor, liver = vols[ph]
        center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)        # Week 2: each phase its own centre
        out.append(np.clip(adaptive_crop(ct_masked, center, side, False), 0, 255).astype(np.uint8))
    return pid, np.stack(out), side

t0 = time.time()
CT_ADAPT, SIDE_INT = {}, {}
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    for n, (pid, arr, side) in enumerate(pool.map(internal_adaptive, dev_ids + test_ids), start=1):
        CT_ADAPT[pid], SIDE_INT[pid] = arr, side
        if n % 50 == 0:
            print(f'  internal adaptive crops {n}/278 ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)
small = [p for p in CT_ADAPT if SIDE_INT[p] == 96]
identical = [p for p in small if np.array_equal(CT_ADAPT[p], FIXED[p])]
GATE_INTERNAL = {'patients_side_96': len(small), 'identical_to_processed': len(identical), 'patients_side_above_96': len(CT_ADAPT) - len(small)}
print('Internal adaptive crops:', GATE_INTERNAL, '| side by class (median):',
      pd.Series(SIDE_INT).groupby(lambda p: CLASSES[LABEL[p]]).median().to_dict())
if IN_COLAB and len(identical) < 0.99 * len(small):
    raise RuntimeError('STOP — adaptive crops of small tumours differ from the processed crops. Send this output.')
pd.DataFrame({'patient_id': list(SIDE_INT), 'side': list(SIDE_INT.values())}).to_csv(OUT_DIR / 'E4d_internal_adaptive_sides.csv', index=False)
CT = CT_ADAPT                                   # the training code below uses CT

In [ ]:
# Cell 6 — Training code: E3 phase-dropout protocol, plus scanner-variation augmentation and the no-non-contrast option
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):                            # Week 4 / E3, unchanged
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept(allowed):
    """E3 phase dropout over the allowed channels: all allowed with p = 0.5, else drop each with p = 0.5, keep >= 1."""
    if random.random() < 0.5:
        return tuple(allowed)
    while True:
        kept = tuple(c for c in allowed if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

def domain_randomise(raw4):
    """Scanner-variation augmentation (plan 21, E8b-2) on the 0-255 crop before normalisation; one draw for all phases."""
    x = raw4.astype(np.float32)
    mask = x > 0
    if random.random() < 0.5:
        sigma = random.uniform(0.0, 1.5)
        x = np.stack([ndimage.gaussian_filter(x[c], sigma) for c in range(4)])
    if random.random() < 0.5:
        x = x + np.random.normal(0.0, random.uniform(0.0, 10.0), x.shape).astype(np.float32)
    if random.random() < 0.5:
        gamma = random.uniform(0.7, 1.5)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** gamma
    if random.random() < 0.5:
        f = random.uniform(0.5, 1.0)
        down = [ndimage.zoom(x[c], (f, f, 1.0), order=1) for c in range(4)]
        x = np.stack([ndimage.zoom(d, (96 / d.shape[0], 96 / d.shape[1], 1.0), order=1)[:96, :96, :96] for d in down])
    if random.random() < 0.5:
        sk = random.uniform(0.0, 2.0)
        x = np.stack([ndimage.gaussian_filter(x[c], (0, 0, sk)) for c in range(4)])
    if random.random() < 0.5:
        x = x * random.uniform(0.9, 1.1)
    return (np.clip(x, 0, 255) * mask).astype(np.float32)

class CropDataset(Dataset):
    def __init__(self, ids, arrays, train=False, phase_dropout=False, dr=False, allowed=(0, 1, 2, 3), kept=None):
        self.ids, self.arrays, self.train, self.phase_dropout, self.dr = list(ids), arrays, train, phase_dropout, dr
        self.allowed, self.kept = tuple(allowed), tuple(kept) if kept is not None else tuple(allowed)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        raw = self.arrays[pid].astype(np.float32)
        if self.train and self.dr:
            raw = domain_randomise(raw)
        vol4 = zscore4(raw)
        if self.train:
            vol4 = augment(vol4)
        kept = sample_kept(self.allowed) if (self.train and self.phase_dropout) else self.kept
        if kept != (0, 1, 2, 3):
            vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL.get(pid, 0), pid

def seed_worker(worker_id):                  # each loader worker gets its own numpy stream (torch seeds python/torch per worker)
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, arrays, kept=(0, 1, 2, 3), batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids, arrays, kept=kept), batch_size=batch_size, shuffle=False,
                        num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)

def train_fold(arm, cfg, fold):
    path = OUT_DIR / arm / f'fold_{fold}.pt'
    if path.exists():
        print(f'  {arm} fold {fold}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(fold_splits[fold]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + fold,
                                              stratify=[LABEL[p] for p in outer_train])
    set_seeds(SEED + fold)
    loader = DataLoader(CropDataset(inner_train, CT, train=True, phase_dropout=True, dr=cfg['dr'], allowed=cfg['allowed']),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=USE_AMP, drop_last=True,
                        persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch, history = -1.0, None, -1, []
    for epoch in range(NUM_EPOCHS):
        model.train()
        started, loss_sum, n = time.time(), 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        inner_auc = macro_auc(y_inner, predict(model, inner_val, CT, kept=cfg['allowed']).loc[inner_val].to_numpy())
        history.append({'epoch': epoch, 'train_loss': loss_sum / max(n, 1), 'inner_val_auc': inner_auc, 'seconds': time.time() - started})
        if inner_auc > best_auc:
            best_auc, best_epoch = inner_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'arm': arm, 'fold': fold, 'best_epoch': best_epoch, 'best_inner_val_auc': best_auc,
                'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, 'epochs': NUM_EPOCHS}, path)
    pd.DataFrame(history).to_csv(OUT_DIR / arm / f'fold_{fold}_history.csv', index=False)
    print(f'  {arm} fold {fold}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} ({sum(h["seconds"] for h in history) / 60:.1f} min)', flush=True)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

In [ ]:
# Cell 13 — Train the adaptive-crop phase-dropout CNN (5 folds, E3 protocol)
ARM_CFG = {'dr': False, 'allowed': (0, 1, 2, 3)}
t0 = time.time()
for fold in range(5):
    train_fold('adaptive_phase_dropout', ARM_CFG, fold)
print(f'Training done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 9 — Evaluation (internal, by size, MCT-LTDiag 203), fusions with E2 radiomics, saved outputs (HCC-TACE-Seg follows at the end)
PATTERNS = {'complete': (0, 1, 2, 3), 'drop_P': (1, 2, 3), 'drop_C1': (0, 2, 3), 'drop_C2': (0, 1, 3), 'drop_C3': (0, 1, 2),
            'P+C2': (0, 2), 'C2_only': (2,)}
ADAPT_MODELS = [load_state(OUT_DIR / 'adaptive_phase_dropout' / f'fold_{f}.pt') for f in range(5)]
D_MODELS = [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)]
def oof_and_test(models, arrays, kept):
    oof = pd.concat([predict(models[f], fold_splits[f]['val_ids'], arrays, kept=kept) for f in range(5)]).loc[dev_ids]
    test = sum(predict(m, test_ids, arrays, kept=kept).loc[test_ids] for m in models) / 5
    return oof, test
def external(models, arrays, ids):
    return sum(predict(m, ids, arrays).loc[ids] for m in models) / 5

INTERNAL = {('adaptive_CNN', p): oof_and_test(ADAPT_MODELS, CT_ADAPT, k) for p, k in PATTERNS.items()}
d_saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
for p in PATTERNS:
    t = d_saved[d_saved['pattern'] == p]
    INTERNAL[('phase_dropout_CNN', p)] = (t[t['split'] == 'oof'].set_index('patient_id').loc[dev_ids, P3], t[t['split'] == 'test'].set_index('patient_id').loc[test_ids, P3])
EXT = {'adaptive_CNN': (external(ADAPT_MODELS, MCT_ADAPT, MCT_IDS), None),          # HCC-TACE-Seg is added in the last cell
       'phase_dropout_CNN': (external(D_MODELS, MCT_PRIM, MCT_IDS), None)}
e80 = pd.read_csv(E8A_RUN / 'E8_0_E6_203_predictions.csv', dtype={'patient_id': str})
e80 = e80[e80['variant'] == 'primary']
GATES = {'mct_crops': GATE_MCT_CROPS, 'internal_crops': GATE_INTERNAL,
         'MCT_D_vs_E8_0': float(np.abs(EXT['phase_dropout_CNN'][0].to_numpy() - e80[e80['model'] == 'phase_dropout_CNN'].set_index('patient_id').loc[MCT_IDS, P3].to_numpy()).max())}
print('Gates:', GATES)
if IN_COLAB and GATES['MCT_D_vs_E8_0'] > 0.02:
    raise RuntimeError('STOP — the fixed-crop CNN does not reproduce E8.0/E5c. Send this output.')

# ---- fusions with E2 radiomics (meta-learners on internal out-of-fold predictions) ----
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
test_meta = pd.read_csv(E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', dtype={'patient_id': str}).set_index('patient_id').loc[test_ids]
RAD_MCT = e80[e80['model'] == 'radiomics_noshape'].set_index('patient_id').loc[MCT_IDS, P3]
def prepare(X):
    X = X.astype(np.float32); X[X <= -1] = np.nan; return X
def meta_folds(columns, oof):
    fitted, oof_p = [], np.full((len(oof), 3), np.nan)
    for f in range(5):
        tr, va = oof['fold'].astype(int).ne(f).to_numpy(), oof['fold'].astype(int).eq(f).to_numpy()
        X = prepare(oof.loc[tr, columns].to_numpy())
        imp = SimpleImputer(strategy='median').fit(X); sc = StandardScaler().fit(imp.transform(X))
        m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof.loc[tr, 'label'].astype(int))
        oof_p[va] = m.predict_proba(sc.transform(imp.transform(prepare(oof.loc[va, columns].to_numpy()))))
        fitted.append((imp, sc, m))
    return fitted, pd.DataFrame(oof_p, index=oof['patient_id'], columns=P3)
def apply_meta(fitted, frame, columns):
    return pd.DataFrame(np.mean([m.predict_proba(s.transform(i.transform(prepare(frame[columns].to_numpy())))) for i, s, m in fitted], axis=0), index=frame.index, columns=P3)
FUS, FITTED_META = {}, {}
for cnn in ['adaptive_CNN', 'phase_dropout_CNN']:
    c_oof, c_test = INTERNAL[(cnn, 'complete')]
    oof = oof_meta.copy(); oof[W4] = c_oof.loc[oof['patient_id']].to_numpy()
    frames = {'test': test_meta.copy(), 'MCT': pd.DataFrame(index=MCT_IDS)}
    frames['test'][W4] = c_test.loc[test_ids].to_numpy()
    frames['MCT'][W4], frames['MCT'][W5] = EXT[cnn][0].to_numpy(), RAD_MCT.to_numpy()
    sel = SEL.set_index('patient_id').loc[MCT_IDS]
    frames['MCT']['age'] = sel['patient_age'].str.extract(r'0*(\d+)')[0].astype(float).to_numpy()
    frames['MCT']['gender'] = sel['patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()
    for kind, cols in {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}.items():
        fitted, oof_p = meta_folds(cols, oof)
        FUS[(kind, cnn)] = (oof_p.loc[dev_ids], apply_meta(fitted, frames['test'], cols), apply_meta(fitted, frames['MCT'], cols), None)
        FITTED_META[(kind, cnn)] = (fitted, cols)                    # kept for HCC-TACE-Seg in the last cell
ref = {k: round(macro_auc([LABEL[p] for p in test_ids], FUS[(k, 'phase_dropout_CNN')][1].to_numpy()), 3) for k in ['cnn_plus_radiomics', 'full_fusion']}
GATES['reference_fusions_internal'] = ref
print('Reference fusions with the fixed-crop CNN (must equal E3: 0.952, 0.951):', ref)

# ---- metrics ----
def compute_midrank(values):
    order = np.argsort(values); sorted_values = values[order]; n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64); index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1; index = end
    output = np.empty(n_values, dtype=np.float64); output[order] = midranks
    return output
def delong_components(y, s):
    pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    return tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m
def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0, auc - 1.96 * se)), float(min(1, auc + 1.96 * se))
def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1); a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))
y_test = np.array([LABEL[p] for p in test_ids]); y_dev = np.array([LABEL[p] for p in dev_ids])
Y_MCT = (SEL.set_index('patient_id').loc[MCT_IDS, 'type'] == 'HCC').astype(int).to_numpy()
rng = np.random.default_rng(SEED)
BOOT_T = [rng.integers(0, len(test_ids), len(test_ids)) for _ in range(2000)]
BOOT_M = [rng.integers(0, len(MCT_IDS), len(MCT_IDS)) for _ in range(2000)]
CUTS = np.percentile([SIZE[p] for p in dev_ids], [100 / 3, 200 / 3])
THIRD = {p: 'small' if SIZE[p] <= CUTS[0] else 'medium' if SIZE[p] <= CUTS[1] else 'large' for p in dev_ids + test_ids}
ITEMS_INT = {f'CNN {c} [{p}]': v for (c, p), v in INTERNAL.items()}
ITEMS_INT.update({f'{k}: {c}': (v[0], v[1]) for (k, c), v in FUS.items()})
rows, strata = [], []
for name, (oof, test) in ITEMS_INT.items():
    p = test.loc[test_ids].to_numpy(); pred = p.argmax(axis=1)
    vals = [macro_auc(y_test[b], p[b]) for b in BOOT_T if len(np.unique(y_test[b])) == 3]
    fold = [macro_auc([LABEL[x] for x in s['val_ids']], oof.loc[s['val_ids']].to_numpy()) for s in fold_splits]
    rows.append({'model': name, 'test_macro_auc': macro_auc(y_test, p), 'boot_low': float(np.percentile(vals, 2.5)), 'boot_high': float(np.percentile(vals, 97.5)),
                 'cv_mean': float(np.mean(fold)), 'cv_sd': float(np.std(fold)),
                 **{f'recall_{c}': float(((pred == i) & (y_test == i)).sum() / (y_test == i).sum()) for i, c in enumerate(CLASSES)}})
    if name.endswith('[complete]') or name.startswith(('cnn_plus', 'full')):
        for g in ['small', 'medium', 'large']:
            ids = [x for x in dev_ids if THIRD[x] == g]
            yy = np.array([LABEL[x] for x in ids], dtype=int); pp = oof.loc[ids].to_numpy()
            strata.append({'model': name, 'third': g, 'n': len(ids), 'oof_macro_auc': macro_auc(yy, pp) if np.bincount(yy, minlength=3).min() >= 3 else np.nan,
                           'oof_accuracy': float((pp.argmax(1) == yy).mean()),
                           **{f'recall_{c}': float(((pp.argmax(1) == i) & (yy == i)).sum() / max(1, (yy == i).sum())) for i, c in enumerate(CLASSES)}})
INTERNAL_RESULTS, STRATA = pd.DataFrame(rows), pd.DataFrame(strata)
ITEMS_EXT = {f'CNN {c}': v for c, v in EXT.items()}
ITEMS_EXT.update({f'{k}: {c}': (v[2], v[3]) for (k, c), v in FUS.items()})
mrows = []
for name, (mct, tace) in ITEMS_EXT.items():
    pm = mct.loc[MCT_IDS].to_numpy(); s = pm[:, 0] / np.clip(pm[:, 0] + pm[:, 1], 1e-12, None); pred = pm.argmax(1)
    auc, lo, hi = delong_ci(Y_MCT, s)
    big = np.array([MCT_SIDE[p] > 96 for p in MCT_IDS])
    mrows.append({'model': name, 'auc': auc, 'delong_low': lo, 'delong_high': hi, 'hcc_sensitivity': float(((pred == 0) & (Y_MCT == 1)).sum() / Y_MCT.sum()),
                  'icc_specificity': float(((pred == 1) & (Y_MCT == 0)).sum() / (1 - Y_MCT).sum()),
                  'auc_large_tumours_side_above_96': float(roc_auc_score(Y_MCT[big], s[big])) if 0 < Y_MCT[big].sum() < big.sum() else np.nan, 'n_large': int(big.sum())})
MCT_RESULTS = pd.DataFrame(mrows)
PAIRED = []
for a, b in [('CNN adaptive_CNN', 'CNN phase_dropout_CNN'), ('cnn_plus_radiomics: adaptive_CNN', 'cnn_plus_radiomics: phase_dropout_CNN'),
             ('full_fusion: adaptive_CNN', 'full_fusion: phase_dropout_CNN')]:
    sa = ITEMS_EXT[a][0].loc[MCT_IDS].to_numpy(); sb = ITEMS_EXT[b][0].loc[MCT_IDS].to_numpy()
    sa, sb = sa[:, 0] / (sa[:, 0] + sa[:, 1]), sb[:, 0] / (sb[:, 0] + sb[:, 1])
    diffs = [roc_auc_score(Y_MCT[i], sa[i]) - roc_auc_score(Y_MCT[i], sb[i]) for i in BOOT_M if 0 < Y_MCT[i].sum() < len(i)]
    ta, tb = ITEMS_INT[a.replace('CNN adaptive_CNN', 'CNN adaptive_CNN [complete]').replace('CNN phase_dropout_CNN', 'CNN phase_dropout_CNN [complete]')][1].to_numpy(), \
             ITEMS_INT[b.replace('CNN phase_dropout_CNN', 'CNN phase_dropout_CNN [complete]')][1].to_numpy()
    tdiffs = [macro_auc(y_test[i], ta[i]) - macro_auc(y_test[i], tb[i]) for i in BOOT_T if len(np.unique(y_test[i])) == 3]
    PAIRED.append({'model': a, 'reference': b, 'mct_difference': roc_auc_score(Y_MCT, sa) - roc_auc_score(Y_MCT, sb),
                   'mct_boot_low': float(np.percentile(diffs, 2.5)), 'mct_boot_high': float(np.percentile(diffs, 97.5)), 'mct_delong_p': delong_paired_p(Y_MCT, sa, sb),
                   'internal_test_difference': macro_auc(y_test, ta) - macro_auc(y_test, tb),
                   'internal_boot_low': float(np.percentile(tdiffs, 2.5)), 'internal_boot_high': float(np.percentile(tdiffs, 97.5))})
PAIRED = pd.DataFrame(PAIRED)
for name, frame in [('E4d_internal_results', INTERNAL_RESULTS), ('E4d_internal_by_size_third', STRATA), ('E4d_mct203_results', MCT_RESULTS),
                    ('E4d_paired_differences', PAIRED)]:
    frame.to_csv(OUT_DIR / f'{name}.csv', index=False)
summary = {'run_id': RUN_ID, 'notebook': 'E4d_adaptive_crop_CNN_v3.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes, 'gates': GATES,
           'epochs': NUM_EPOCHS, 'size_cutoffs_dev_voxels': CUTS.tolist(), 'adaptive_margin': ADAPT_MARGIN,
           'protocol': '00 log section 19, E4d (approved 17 Oct 2026); the fixed-crop phase-dropout CNN remains the main CNN'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file() and p.suffix != '.npz']
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E4d_{RUN_ID} | epochs {NUM_EPOCHS} | gates: {GATES}')
print('\n--- internal held-out 56 ---')
print(INTERNAL_RESULTS.round(3).to_string(index=False))
print('\n--- internal out-of-fold by size third ---')
print(STRATA.round(3).to_string(index=False))
print('\n--- MCT-LTDiag 203 ---')
print(MCT_RESULTS.round(3).to_string(index=False))
print('\n--- paired differences (adaptive vs fixed crop) ---')
print(PAIRED.round(3).to_string(index=False))
print('(HCC-TACE-Seg follows in the last cell)')
print('==============================================================')

pred_rows = []                                   # per-patient predictions, for figures and the last cell
for (c, p), (oof, test) in INTERNAL.items():
    if p == 'complete':
        for split_name, frame in [('internal_oof', oof), ('internal_test', test)]:
            pred_rows.append(frame.assign(model=f'CNN {c}', cohort=split_name).rename_axis('patient_id').reset_index())
for c, (mct, _) in EXT.items():
    pred_rows.append(mct.assign(model=f'CNN {c}', cohort='MCT_LTDiag').rename_axis('patient_id').reset_index())
for (k, c), (oof, test, mct, _) in FUS.items():
    for split_name, frame in [('internal_oof', oof), ('internal_test', test), ('MCT_LTDiag', mct)]:
        pred_rows.append(frame.assign(model=f'{k}: {c}', cohort=split_name).rename_axis('patient_id').reset_index())
PREDICTIONS = pd.concat(pred_rows, ignore_index=True)
PREDICTIONS.to_csv(OUT_DIR / 'E4d_predictions.csv', index=False)
print('Main results saved; HCC-TACE-Seg next.')

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder (E5's cell; output folder E5c_<time>)
import os, sys, json, hashlib, platform, datetime, re, time, warnings, subprocess, shutil, tarfile
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'openpyxl', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E5_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
from concurrent.futures import ThreadPoolExecutor
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy.ndimage import binary_dilation
from scipy.stats import beta

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
EXT_DIR = PROJECT_DIR / 'external_validation' / 'HCC-TACE-Seg'
OLD_PREFIX = os.environ['LIVER_HISTORICAL_PREFIX']   # paths stored in the Week-10 manifest (folder moved 7 Sep 2026)
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']

INPUTS = {
    'manifest_v4': (EXT_DIR / 'nifti_v4_liver_adapter' / 'manifests' / 'external_manifest_for_week10_tumor_only_FILTERED_liver_adapter.csv',
                    'e428d3ad7b5928622e51b78c1c3346b9b168ef9e6b6205b00e33a61f9f84e5ad'),
    'dicom_catalog': (EXT_DIR / 'nifti' / 'logs' / 'dicom_catalog.csv', '515453cc76f824b0803a4d885e0065e395198e4ba3521c2df79b21762e395e62'),
    'historical_external': (PROJECT_DIR / 'results' / 'external_HCC_TACE_SEG_liver_adapter_fixed_103' / 'external_predictions_with_phase_audit.csv',
                            'a5e412a63778013130c90bb50ae37591e16e4d85964053d15abf83b1443424fb'),
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e5_v2_predictions': (REVISION_DIR / 'E5_20261017T062958Z' / 'E5_predictions.csv', '5100abe40d584c69872099121175d71d13a67ab407e684f8ae497391e12c1149'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, lightgbm, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'lightgbm': lightgbm.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE)}
RUN_ID = globals().get('RUN_ID') or datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')   # same run as Cell 1
OUT_DIR = REVISION_DIR / f'E4d_{RUN_ID}'
N_IO = 4 if IN_COLAB else 1
N_JOBS = os.cpu_count() if IN_COLAB else 1
OUT_DIR.mkdir(parents=False, exist_ok=True)
(OUT_DIR / 'inputs').mkdir(exist_ok=True)
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Download the official TCIA clinical file and extract age and sex (stops if the format is unclear)
CLINICAL_URL = 'https://www.cancerimagingarchive.net/wp-content/uploads/HCC-TACE-Seg_clinical_data-V2.xlsx'
if IN_COLAB:
    import urllib.request
    clinical_path = OUT_DIR / 'inputs' / 'HCC-TACE-Seg_clinical_data-V2.xlsx'
    request = urllib.request.Request(CLINICAL_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=120) as response:
        clinical_path.write_bytes(response.read())
    sheets = pd.read_excel(clinical_path, sheet_name=None)
else:
    clinical_path = OUT_DIR / 'inputs' / 'clinical_test.csv'
    shutil.copy2(os.environ['E5_TEST_CLINICAL'], clinical_path)
    sheets = {'test': pd.read_csv(clinical_path)}
    if os.environ.get('E5_TEST_GLOSSARY'):
        sheets['glossary'] = pd.read_csv(os.environ['E5_TEST_GLOSSARY'])
input_hashes['tcia_clinical'] = sha256_of(clinical_path)

def normalized(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()

def find_clinical_table(sheets):
    for sheet_name, table in sheets.items():
        for column in table.columns:
            values = table[column].dropna().astype(str).str.strip()
            if len(values) and values.str.fullmatch(r'HCC_\d+').mean() > 0.9:
                return sheet_name, table, column
    return None, None, None

sheet_name, table, id_col = find_clinical_table(sheets)
age_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\bage\b', normalized(c))]
sex_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\b(sex|gender)\b', normalized(c))]
print('Sheets:', list(sheets))
if table is not None:
    print('Using sheet:', sheet_name, '| ID column:', id_col, '| age columns:', age_cols, '| sex columns:', sex_cols)
if table is None or len(age_cols) != 1 or len(sex_cols) != 1:
    for name, t in sheets.items():
        print(f'--- sheet {name}: columns', list(t.columns))
    raise RuntimeError('STOP — could not identify a unique patient-ID, age and sex column. '
                       'Send this output; the column choice will be set explicitly.')

SEX_MAP = {'m': 1.0, 'male': 1.0, 'f': 0.0, 'female': 0.0}     # internal coding: male = 1, female = 0

def sex_tokens(series):
    return series.astype(str).str.strip().str.lower().str.replace(r'^(\d+)\.0$', r'\1', regex=True)

def sex_codes_from_glossary(sheets, sex_col):
    """Numeric codes defined in the file's own glossary sheet (TCIA V2: 'Sex | 1=Male, 2=Female')."""
    for name, t in sheets.items():
        if name == sheet_name:
            continue
        for row in [list(t.columns)] + t.astype(str).values.tolist():
            cells = [str(v).strip() for v in row]
            if cells and normalized(cells[0]) == normalized(sex_col):
                text = ' '.join(cells[1:])
                pairs = re.findall(r'(\d+)\s*=\s*(male|female)\b', text, flags=re.I)
                if pairs:
                    return {code: SEX_MAP[word.lower()] for code, word in pairs}, f'sheet {name!r}: {cells[0]} | {text}'
    return {}, None

raw_sex = sex_tokens(table[sex_cols[0]].dropna())
print('Sex values in the file:', raw_sex.value_counts().to_dict())
SEX_CODING = 'text (male/female)'
if set(raw_sex) - set(SEX_MAP):
    glossary_codes, glossary_line = sex_codes_from_glossary(sheets, sex_cols[0])
    print('Sex coding stated in the file glossary:', glossary_line)
    if sorted(glossary_codes.values()) != [0.0, 1.0]:
        raise RuntimeError('STOP — numeric sex codes and no clear male/female definition in the glossary. Send this output.')
    SEX_MAP = {**SEX_MAP, **glossary_codes}
    SEX_CODING = f'numeric, from the glossary ({glossary_line}); mapped to male = 1, female = 0'
unmapped = sorted(set(raw_sex) - set(SEX_MAP))
if unmapped:
    raise RuntimeError(f'STOP — unrecognised sex codes {unmapped}. Send this output; the coding will be set explicitly.')
mapped = raw_sex.map(SEX_MAP)
print(f'Sex after mapping to the internal coding: male {int((mapped == 1).sum())}, female {int((mapped == 0).sum())}')
raw_age = pd.to_numeric(table[age_cols[0]], errors='coerce')
print(f'Age: {raw_age.notna().sum()} numeric values, range {raw_age.min()}–{raw_age.max()}')
if raw_age.notna().sum() and not (0 < raw_age.min() and raw_age.max() < 110):
    raise RuntimeError('STOP — age values outside 0–110. Send this output.')
clinical = pd.DataFrame({
    'patient_id': table[id_col].astype(str).str.strip().str.replace(r'HCC_(\d+)', lambda m: f'HCC_{int(m.group(1)):03d}', regex=True),
    'age': pd.to_numeric(table[age_cols[0]], errors='coerce'),
    'sex': sex_tokens(table[sex_cols[0]]).map(SEX_MAP),
}).dropna(subset=['patient_id'])
conflicts = clinical.groupby('patient_id').agg(age_n=('age', 'nunique'), sex_n=('sex', 'nunique'))
conflicting_ids = set(conflicts.index[(conflicts['age_n'] > 1) | (conflicts['sex_n'] > 1)])
clinical = clinical.groupby('patient_id', as_index=False).first()
clinical.loc[clinical['patient_id'].isin(conflicting_ids), ['age', 'sex']] = np.nan
print(f'Clinical rows: {len(clinical)} patients; conflicting duplicate rows set to missing: {len(conflicting_ids)}')

In [ ]:
# Cell 3 — External manifest (paths remapped to the new folder) and mixed-scan detection
manifest = pd.read_csv(INPUTS['manifest_v4'][0])
path_cols = [c for c in manifest.columns if re.match(r'(ct|mask|liver_mask)_(P|C1|C2|C3)$', c)]
def remap(value):
    if isinstance(value, str) and value.startswith(OLD_PREFIX):
        return str(PROJECT_DIR) + value[len(OLD_PREFIX):]
    return value
for column in path_cols:
    manifest[column] = manifest[column].map(remap)
referenced = {v for c in path_cols for v in manifest[c].dropna() if isinstance(v, str) and v.strip()}
missing_files = [v for v in referenced if not Path(v).exists()]
assert len(manifest) == 103, len(manifest)
if missing_files:
    raise RuntimeError(f'STOP — {len(missing_files)} manifest files not found, e.g. {missing_files[:3]}')

catalog = pd.read_csv(INPUTS['dicom_catalog'][0], usecols=['SeriesInstanceUID', 'Modality', 'ImagePositionPatient'])
catalog = catalog[catalog['Modality'] == 'CT'].copy()
catalog['z'] = catalog['ImagePositionPatient'].map(
    lambda s: round(float(re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)[2]), 1) if isinstance(s, str) else np.nan)
per_series = catalog.groupby('SeriesInstanceUID').agg(n=('z', 'size'), unique_z=('z', 'nunique'))
per_series['repeats'] = per_series['n'] / per_series['unique_z']

def series_uid(path):
    match = re.search(r'__([0-9.]+)__CT', str(path))
    return match.group(1) if match else None

mixed_rows = []
for _, row in manifest.iterrows():
    for phase in PHASE_ORDER:
        uid = series_uid(row.get(f'ct_{phase}'))
        if uid and uid in per_series.index and per_series.loc[uid, 'repeats'] >= 1.9:
            mixed_rows.append({'patient_id': row['patient_id'], 'channel': phase, 'series_uid': uid,
                               'files': int(per_series.loc[uid, 'n']), 'unique_positions': int(per_series.loc[uid, 'unique_z']),
                               'repeats': float(per_series.loc[uid, 'repeats'])})
mixed = pd.DataFrame(mixed_rows)
MIXED_IDS = set(mixed['patient_id']) if len(mixed) else set()
ALL_IDS = manifest['patient_id'].astype(str).tolist()
CLEAN_IDS = [p for p in ALL_IDS if p not in MIXED_IDS]
mixed.to_csv(OUT_DIR / 'E5_mixed_scan_inputs.csv', index=False)
print(f'Mixed-scan inputs: {len(mixed)} in {len(MIXED_IDS)} patients -> clean cohort {len(CLEAN_IDS)} patients')
if IN_COLAB and (len(mixed), len(MIXED_IDS)) != (16, 14):
    raise RuntimeError('STOP — mixed-scan count differs from the 17 Oct audit (16 inputs, 14 patients). Send this output.')

clinical_ext = pd.DataFrame({'patient_id': ALL_IDS}).merge(clinical, on='patient_id', how='left')
print(f'Observed age for {clinical_ext["age"].notna().sum()}/103, observed sex for {clinical_ext["sex"].notna().sum()}/103 '
      f'(male {int((clinical_ext["sex"] == 1).sum())}, female {int((clinical_ext["sex"] == 0).sum())})')
clinical_ext.to_csv(OUT_DIR / 'E5_external_clinical_joined.csv', index=False)

In [ ]:
# Cell 4 — Preprocessing: E5/Week-10g code for the gates, plus the axis-fixed variants (00 log 18)
TARGET_SPACING_MM = (1.0, 1.0, 1.0)
CROP_SIZE = (96, 96, 96)
TUMOR_DILATION_MM = 15
INPLANE_MM = 400.0 / 512                      # internal physical pixel size (E6 primary rule)

def resample_to_isotropic(arr, original_spacing, target_spacing=TARGET_SPACING_MM, is_mask=False):   # Week 10g
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(original_spacing))
    new_size = [int(round(sz * os_ / ts_)) for sz, os_, ts_ in zip(img.GetSize(), img.GetSpacing(), target_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0 if is_mask else -1024)
    out = sitk.GetArrayFromImage(resampler.Execute(img))
    return out.astype(np.uint8) if is_mask else out.astype(np.float32)

def window_historical(hu):          # Week 10g: W150/C50 -> -25..125 HU -> 0..255
    lo, hi = 50 - 150 / 2, 50 + 150 / 2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

def window_corrected(hu):           # PLC-CECT release convention: -200..200 HU -> 0..255
    lo, hi = -200.0, 200.0
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

WINDOWS = {'historical_W150C50': window_historical, 'corrected_-200_200': window_corrected}

def crop_96_around_tumor(vol, tumor_union, size=CROP_SIZE):   # Week 10g
    if tumor_union.sum() == 0:
        cz, cy, cx = vol.shape[0] // 2, vol.shape[1] // 2, vol.shape[2] // 2
    else:
        cz, cy, cx = np.argwhere(tumor_union > 0).mean(axis=0).astype(int)
    dz, dy, dx = [s // 2 for s in size]
    out = np.zeros(size, dtype=vol.dtype)
    z0, z1 = max(0, cz - dz), min(vol.shape[0], cz + dz)
    y0, y1 = max(0, cy - dy), min(vol.shape[1], cy + dy)
    x0, x1 = max(0, cx - dx), min(vol.shape[2], cx + dx)
    oz0 = dz - (cz - z0); oy0 = dy - (cy - y0); ox0 = dx - (cx - x0)
    out[oz0:oz0 + (z1 - z0), oy0:oy0 + (y1 - y0), ox0:ox0 + (x1 - x0)] = vol[z0:z1, y0:y1, x0:x1]
    return out, (cz, cy, cx)

def zscore_nonzero(x):                                          # Week 10g / Week 4
    mask = x > 0
    if mask.sum() < 100:
        return x.astype(np.float32)
    vals = x[mask]
    out = np.zeros_like(x, dtype=np.float32)
    out[mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def read_nifti(path):
    if not isinstance(path, str) or not path.strip() or path.strip().lower() in {'nan', 'none', 'null'}:
        return None, None
    img = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32), img.GetSpacing()

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

# ---- orientation (E6 rule, 00 log 17.9): internal reference from the internal non-contrast scans ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254

def orientation_descriptor(ct0255_ijk, liver_ijk):
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

def decide(d):
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': bool(np.isfinite(off_i) and np.isfinite(off_j)
                                                              and max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03)}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')
def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
ok_int = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok_int['bone_j'] - .5) > abs(ok_int['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok_int['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok_int['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok_int['bulk_k'] < .5).mean()), 'n': int(len(ok_int))}
print('Internal orientation reference:', REF)
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')
# Expected orientation of these files (Week 10b wrote them with SimpleITK from DICOM: LPS, slices feet -> head); listed for the summary
EXPECTED = {'transpose': False, 'flip_i': False, 'flip_j': False, 'flip_k': True}

# ---- per patient: E5 outputs (gates) and the axis-fixed variants from the same Week-10g arrays ----
def load_resampled(row, target_for):
    raw, spacing = {}, None
    for phase in PHASE_ORDER:
        ct, sp = read_nifti(row.get(f'ct_{phase}'))
        if ct is not None and spacing is None:
            spacing = sp
        raw[phase] = (ct, read_nifti(row.get(f'mask_{phase}'))[0], read_nifti(row.get(f'liver_mask_{phase}'))[0])
    spacing = spacing or (1.0, 1.0, 1.0)
    target = target_for(spacing)
    ct_r, tm_r, lm_r = {}, {}, {}
    for phase, (ct, tm, lm) in raw.items():
        ct_r[phase] = None if ct is None else resample_to_isotropic(ct, spacing, target)
        tm_r[phase] = None if (ct is None or tm is None) else resample_to_isotropic((tm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
        lm_r[phase] = None if (ct is None or lm is None) else resample_to_isotropic((lm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
    masks_present = [m for m in tm_r.values() if m is not None]
    if not masks_present:
        return None
    ref_shape = masks_present[0].shape
    tumor_union = np.zeros(ref_shape, dtype=np.uint8)
    for m in masks_present:
        if m.shape == ref_shape:
            tumor_union = np.maximum(tumor_union, m)
    liver, liver_fallback = None, False
    for phase in ['C2', 'C1', 'P', 'C3']:
        lm = lm_r.get(phase)
        if lm is not None and lm.shape == ref_shape and lm.sum() > 0:
            liver = lm
            break
    if liver is None:
        liver, liver_fallback = binary_dilation(tumor_union > 0, iterations=TUMOR_DILATION_MM).astype(np.uint8), True
    return {'ct': ct_r, 'tm': tm_r, 'ref_shape': ref_shape, 'tumor_union': tumor_union, 'liver': liver, 'liver_fallback': liver_fallback}

def e5_outputs(r):                                              # E5 cell 4 loop, unchanged
    ct_r, tm_r, ref_shape, tumor_union, liver = r['ct'], r['tm'], r['ref_shape'], r['tumor_union'], r['liver']
    out = {'tumor_voxels': int(tumor_union.sum()), 'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8)}
    for window_name, window_fn in WINDOWS.items():
        cnn = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        rad = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        available = []
        for i, phase in enumerate(PHASE_ORDER):
            if ct_r[phase] is None or ct_r[phase].shape != ref_shape:
                continue
            crop, _ = crop_96_around_tumor(window_fn(ct_r[phase]) * (liver > 0), tumor_union)
            rad[i] = crop
            cnn[i] = zscore_nonzero(crop)
            available.append(phase)
            if tm_r[phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(tm_r[phase], tumor_union)[0]
        out[window_name] = {'cnn': cnn} if window_name == 'historical_W150C50' else {'cnn': cnn, 'rad': rad}
        out['available'] = available
    return out

def descriptor_of(r):
    for phase in ['P', 'C2', 'C1', 'C3']:                         # non-contrast first (bone not confused with contrast)
        if r['ct'][phase] is not None and r['ct'][phase].shape == r['ref_shape']:
            to_ijk = lambda a: np.transpose(a, (2, 1, 0))
            d = orientation_descriptor(window_corrected(to_ijk(r['ct'][phase])), to_ijk(r['liver']) > 0)
            if d is not None:
                d['phase_used'] = phase
            return d
    return None

def axis_fixed(r, o, rules):
    """(k, j, i) Week-10g arrays -> (i, j, k), matched to the internal orientation; Week-10g or Week-2 crop rules."""
    to = lambda a: apply_orientation(np.transpose(a, (2, 1, 0)), o)
    tu, lv = to(r['tumor_union']), to(r['liver'])
    out = {'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8), 'cnn': np.zeros((4, *CROP_SIZE), np.float32),
           'rad': np.zeros((4, *CROP_SIZE), np.float32 if rules == 'iso1mm' else np.uint8)}
    crop = None
    center = compute_center_of_mass(tu) or compute_center_of_mass(lv) or tuple(s // 2 for s in tu.shape)
    SIDE = adaptive_side([tu > 0])
    out['side'] = SIDE
    for i, phase in enumerate(PHASE_ORDER):
        ct = r['ct'][phase]
        if ct is None or ct.shape != r['ref_shape']:
            continue
        if rules == 'iso1mm':                                   # E5 rules, only the axes changed
            crop, _ = crop_96_around_tumor(window_corrected(to(ct)) * (lv > 0), tu)
            if r['tm'][phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(to(r['tm'][phase]), tu)[0]
        elif rules == 'E6rules':                                # E6 primary rules: Week-2 crop, 0-255 uint8
            crop = np.clip(crop_cube(window_corrected(to(ct)) * (lv > 0), center, CROP_SIZE[0]), 0, 255).astype(np.uint8)
            if r['tm'][phase] is not None:
                out['tumor'][i] = (crop_cube(to(r['tm'][phase]), center, CROP_SIZE[0]) > 0).astype(np.uint8)
        if rules == 'adaptive':                                   # E4d: adaptive crop on the E6-rules geometry
            crop = np.clip(adaptive_crop(window_corrected(to(ct)) * (lv > 0), center, SIDE, False), 0, 255).astype(np.uint8)
            if r['tm'][phase] is not None:
                out['tumor'][i] = (adaptive_crop(to(r['tm'][phase]).astype(np.float32), center, SIDE, True) > 0).astype(np.uint8)
        out['rad'][i] = crop
        out['cnn'][i] = zscore_nonzero(crop.astype(np.float32))
        out['available'].append(phase)
    out['tumor_in_crop'] = float(out['tumor'].max(axis=0).sum() / max(1, tu.sum()))
    return out

def header_orientation(direction):
    """Amendment 18.1: map index axes to the internal convention (axis 0 -> patient's left, axis 1 -> posterior,
    axis 2 -> feet) from the LPS direction cosines in the image header. None if not an axial acquisition."""
    D = np.array(direction, dtype=float).reshape(3, 3)          # column c = physical direction of index axis c
    p = [int(np.argmax(np.abs(D[:, c]))) for c in range(3)]
    s = [float(np.sign(D[p[c], c])) for c in range(3)]
    if p[2] != 2 or sorted(p[:2]) != [0, 1]:
        return None
    transpose = p[0] == 1
    a0, a1 = (1, 0) if transpose else (0, 1)
    return {'transpose': bool(transpose), 'flip_i': bool(s[a0] < 0), 'flip_j': bool(s[a1] < 0), 'flip_k': bool(s[2] > 0)}

def reference_ct_path(row):                                     # the CT whose spacing Week 10g uses (first available)
    for phase in PHASE_ORDER:
        path = row.get(f'ct_{phase}')
        if isinstance(path, str) and path.strip() and path.strip().lower() not in {'nan', 'none', 'null'}:
            return path
    return None

def header_direction(path):
    reader = sitk.ImageFileReader()
    reader.SetFileName(path)
    reader.ReadImageInformation()
    return reader.GetDirection()

def preprocess(row):
    pid = str(row['patient_id'])
    iso = load_resampled(row, lambda spacing: TARGET_SPACING_MM)
    if iso is None:
        return pid, None
    result = e5_outputs(iso)
    direction = header_direction(reference_ct_path(row))
    o = header_orientation(direction)
    d = descriptor_of(iso)
    anatomy = decide(d) if d is not None and np.isfinite(d['bone_i']) else {}
    result['orientation'] = {'header_axial': o is not None, 'direction': [round(float(v), 3) for v in direction],
                             **(o or {}), **{f'anatomy_{k}': v for k, v in anatomy.items()},
                             **(d or {}), 'liver_fallback': iso['liver_fallback']}
    if o is None:
        return pid, result                                       # stopped below, after all patients are listed
    result['axis_fixed_iso1mm'] = axis_fixed(iso, o, 'iso1mm')
    del iso
    e6 = load_resampled(row, lambda spacing: (INPLANE_MM, INPLANE_MM, float(spacing[2])))
    result['axis_fixed_E6rules'] = axis_fixed(e6, o, 'E6rules')
    result['axis_fixed_adaptive'] = axis_fixed(e6, o, 'adaptive')
    return pid, result

PRE = {}
t0 = time.time()
rows = [row for _, row in manifest.iterrows()]
import faulthandler                       # E4d v3: if the process dies, the Python stack is written to this file
CRASH_LOG = open(OUT_DIR / 'E4d_tace_crash_log.txt', 'w')
faulthandler.enable(file=CRASH_LOG)
TACE_LOG = OUT_DIR / 'E4d_tace_progress.log'     # E4d v3: one patient at a time, logged before and after each patient
for n, row in enumerate(rows, start=1):
    pid = str(row['patient_id'])
    with open(TACE_LOG, 'a') as log:
        log.write(f'start {n} {pid} {time.strftime("%H:%M:%S")}{ram_text()}\n')
    t1 = time.time()
    pid, result = preprocess(row)
    if result is None:
        raise RuntimeError(f'STOP — {pid} could not be preprocessed.')
    PRE[pid] = result
    side = result.get('axis_fixed_adaptive', {}).get('side', '')
    with open(TACE_LOG, 'a') as log:
        log.write(f'done  {n} {pid} {time.time() - t1:.1f} s, adaptive side {side}{ram_text()}\n')
    if n % 10 == 0:
        print(f'  preprocessed {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)
faulthandler.disable(); CRASH_LOG.close()

orient = pd.DataFrame({pid: PRE[pid]['orientation'] for pid in ALL_IDS}).T
orient.to_csv(OUT_DIR / 'E4d_tace_orientation_decisions.csv')
NOT_AXIAL = orient.index[~orient['header_axial'].astype(bool)].tolist()
if NOT_AXIAL:
    raise RuntimeError(f'STOP — image header is not axial for {NOT_AXIAL[:10]}. Send this output.')
ORIENTATION_SUMMARY = {'source': 'image header (amendment 18.1)', 'liver_mask_fallback': int(orient['liver_fallback'].astype(bool).sum())}
for part in EXPECTED:
    applied = orient[part].astype(bool)
    clear = orient[f'anatomy_conf_{part}'].fillna(False).astype(bool) if f'anatomy_conf_{part}' in orient else pd.Series(False, index=orient.index)
    agree = (orient.loc[clear, f'anatomy_{part}'].astype(bool) == applied[clear]) if clear.any() else pd.Series(dtype=bool)
    ORIENTATION_SUMMARY[part] = {'applied_to': int(applied.sum()), 'anatomy_clear': int(clear.sum()),
                                 'anatomy_agrees_with_header': int(agree.sum())}
ORIENTATION_SUMMARY['directions'] = orient['direction'].astype(str).value_counts().head(5).to_dict()
print('Orientation decisions:', json.dumps(ORIENTATION_SUMMARY, indent=1))
TACE_VARIANTS = ['historical_W150C50', 'corrected_-200_200', 'axis_fixed_iso1mm', 'axis_fixed_E6rules', 'axis_fixed_adaptive']
print('Tumour fraction inside the crop (median):', {v: round(float(np.median([PRE[p][v]['tumor_in_crop'] for p in ALL_IDS])), 3)
                                                    for v in TACE_VARIANTS[2:]})

In [ ]:
# Cell 10 — Keep the HCC-TACE-Seg crops (corrected E6 rules and adaptive) in memory
TACE_IDS, TACE_CLEAN = list(ALL_IDS), list(CLEAN_IDS)
TACE_PRIM = {p: PRE[p]['axis_fixed_E6rules']['rad'].astype(np.uint8) for p in TACE_IDS}
TACE_ADAPT = {p: PRE[p]['axis_fixed_adaptive']['rad'].astype(np.uint8) for p in TACE_IDS}
TACE_SIDE = {p: PRE[p]['axis_fixed_adaptive']['side'] for p in TACE_IDS}
TACE_CLIN = clinical_ext.set_index('patient_id').loc[TACE_IDS]
del PRE
print(f'HCC-TACE-Seg crops: {len(TACE_ADAPT)}; adaptive side > 96 for {sum(s > 96 for s in TACE_SIDE.values())} patients')

In [ ]:
# Cell 15 — HCC-TACE-Seg (last): frozen CNNs and the fusions fitted above, applied once; saved outputs
EXT_TACE = {'adaptive_CNN': external(ADAPT_MODELS, TACE_ADAPT, TACE_IDS), 'phase_dropout_CNN': external(D_MODELS, TACE_PRIM, TACE_IDS)}
e5c_saved = pd.read_csv(REVISION_DIR / 'E5c_20261017T143217Z' / 'E5c_predictions.csv', dtype={'patient_id': str})
e5c_saved = e5c_saved[e5c_saved['variant'] == 'axis_fixed_E6rules']
GATES['TACE_D_vs_E5c'] = float(np.abs(EXT_TACE['phase_dropout_CNN'].to_numpy()
                                      - e5c_saved[e5c_saved['model'] == 'phase_dropout_CNN'].set_index('patient_id').loc[TACE_IDS, P3].to_numpy()).max())
print('Gate — fixed-crop CNN vs E5c on HCC-TACE-Seg (max difference, must be < 0.02):', GATES['TACE_D_vs_E5c'])
RAD_TACE = e5c_saved[e5c_saved['model'] == 'radiomics_noshape'].set_index('patient_id').loc[TACE_IDS, P3]
TACE_ITEMS = {f'CNN {c}': v for c, v in EXT_TACE.items()}
for (kind, cnn), (fitted, cols) in FITTED_META.items():
    frame = pd.DataFrame(index=TACE_IDS)
    frame[W4], frame[W5] = EXT_TACE[cnn].to_numpy(), RAD_TACE.to_numpy()
    frame['age'], frame['gender'] = TACE_CLIN['age'].to_numpy(), TACE_CLIN['sex'].to_numpy()
    TACE_ITEMS[f'{kind}: {cnn}'] = apply_meta(fitted, frame, cols)
trows = []
for name, tace in TACE_ITEMS.items():
    for subset, ids in [('clean_89', TACE_CLEAN), ('all_103', TACE_IDS)]:
        tp = tace.loc[ids].to_numpy().argmax(1)
        trows.append({'model': name, 'subset': subset, 'hcc_correct': int((tp == 0).sum()), 'n': len(ids), 'hcc_sensitivity': float((tp == 0).mean())})
TACE_RESULTS = pd.DataFrame(trows)
TACE_RESULTS.to_csv(OUT_DIR / 'E4d_hcc_tace_seg_results.csv', index=False)
PREDICTIONS = pd.concat([PREDICTIONS] + [f.assign(model=n, cohort='HCC_TACE_Seg').rename_axis('patient_id').reset_index() for n, f in TACE_ITEMS.items()], ignore_index=True)
PREDICTIONS.to_csv(OUT_DIR / 'E4d_predictions.csv', index=False)
summary = json.loads((OUT_DIR / 'summary.json').read_text()); summary['gates'] = GATES; summary['hcc_tace_seg'] = 'completed'
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file() and p.suffix != '.npz']
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)
print('\n================ COPY FROM HERE AND SEND BACK (HCC-TACE-Seg part) ================')
print('Gates:', GATES)
print(TACE_RESULTS[TACE_RESULTS['subset'] == 'clean_89'].round(3).to_string(index=False))
print('==================================================================================')

In [ ]:
# Cell 16 — Finish: make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()